# Module 1 — Claim Occurrence Classification with Probability Calibration

## Business Context

We predict **whether a policy will generate a claim** (binary: `HasClaim`). Well-calibrated
probability scores drive **Straight-Through Processing (STP)** routing:

| Lane | Probability | Action |
|------|-------------|--------|
| `auto-settle` | p < 0.10 | Instant payout — no human review |
| `review` | 0.10 ≤ p < 0.40 | Manual adjuster assessment |
| `investigate` | p ≥ 0.40 | Detailed investigation / fraud referral |

We compare **Logistic Regression**, **Random Forest**, and **CatBoost**, then apply
Platt scaling and isotonic calibration, tracking results in **MLflow**.

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from claims.data import load_fremtpl2, build_claims_dataset, split_dataset
from claims.features import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COL,
    get_feature_names,
)
from claims.classification.models import (
    logistic_pipeline,
    random_forest_pipeline,
    catboost_classifier,
    compute_scale_pos_weight,
)
from claims.classification.calibration import (
    calibrate,
    calibration_metrics,
    plot_reliability_diagram,
)
from claims.evaluation import (
    classification_report_dict,
    gini_coefficient,
    stp_routing,
    stp_summary,
)

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
print('Setup complete.')

## 2. Data Preparation

In [ ]:
# Load and join the two freMTPL2 tables
print('Loading freMTPL2 from OpenML...')
freq, sev = load_fremtpl2()
df = build_claims_dataset(freq, sev)
print(f'Dataset shape: {df.shape}')
df.head(3)

In [ ]:
# Stratified train / test split on HasClaim
train_df, test_df = split_dataset(df, target='HasClaim', test_size=0.2, random_state=RANDOM_STATE)
print(f'Train size : {len(train_df):,}  |  Test size: {len(test_df):,}')

# Class imbalance statistics
print()
print('=== Class Imbalance (Train) ===')
y_train_counts = train_df['HasClaim'].value_counts()
print(f"  No claim  (0): {y_train_counts[0]:>8,}  ({y_train_counts[0]/len(train_df)*100:.2f}%)")
print(f"  Has claim (1): {y_train_counts[1]:>8,}  ({y_train_counts[1]/len(train_df)*100:.2f}%)")
print(f"  Imbalance ratio: {y_train_counts[0]/y_train_counts[1]:.1f}:1")

## 3. Feature Matrices

In [ ]:
# Feature and target arrays
all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES
print(f'Numeric features    ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}')
print(f'Categorical features({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}')
print()

# X as DataFrame (required for CatBoost with cat_features as column names)
X_train = train_df[all_features].copy()
X_test  = test_df[all_features].copy()

y_train = train_df[TARGET_COL].values
y_test  = test_df[TARGET_COL].values

print(f'X_train shape: {X_train.shape}  |  X_test shape: {X_test.shape}')
print(f'y_train positives: {y_train.sum():,} / {len(y_train):,}')

## 4. Logistic Regression Baseline

In [ ]:
print('Fitting Logistic Regression...')
lr_pipe = logistic_pipeline(class_weight='balanced', C=1.0)
lr_pipe.fit(X_train, y_train)
print('Done.')

In [ ]:
lr_prob = lr_pipe.predict_proba(X_test)[:, 1]
lr_metrics = classification_report_dict(y_test, lr_prob)

print('=== Logistic Regression — Test Metrics ===')
for k, v in lr_metrics.items():
    print(f'  {k:20s}: {v:.4f}')

## 5. Random Forest

In [ ]:
print('Fitting Random Forest (n_estimators=300, max_depth=8)...')
rf_pipe = random_forest_pipeline(n_estimators=300, max_depth=8, class_weight='balanced')
rf_pipe.fit(X_train, y_train)
print('Done.')

In [ ]:
rf_prob = rf_pipe.predict_proba(X_test)[:, 1]
rf_metrics = classification_report_dict(y_test, rf_prob)

print('=== Random Forest — Test Metrics ===')
for k, v in rf_metrics.items():
    print(f'  {k:20s}: {v:.4f}')

## 6. CatBoost

In [ ]:
# Compute class imbalance weight
spw = compute_scale_pos_weight(y_train)
print(f'scale_pos_weight = {spw:.2f}  (n_neg / n_pos)')

In [ ]:
# CatBoost receives raw string categoricals in a DataFrame
# cat_features passed as list of column names (CatBoost supports this when X is a DataFrame)
print('Fitting CatBoost (iterations=500, lr=0.05, depth=6)...')
cb_model = catboost_classifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    scale_pos_weight=spw,
    verbose=0,
)

# Pass X as DataFrame with raw categorical strings; use column names as cat_features
cb_model.fit(
    X_train,
    y_train,
    cat_features=CATEGORICAL_FEATURES,
    eval_set=(X_test, y_test),
    verbose=False,
)
print('Done.')

In [ ]:
cb_prob = cb_model.predict_proba(X_test)[:, 1]
cb_metrics = classification_report_dict(y_test, cb_prob)

print('=== CatBoost — Test Metrics ===')
for k, v in cb_metrics.items():
    print(f'  {k:20s}: {v:.4f}')

## 7. Probability Calibration

In [ ]:
# Hold out a calibration split from the training set
from sklearn.model_selection import train_test_split as tts

X_cal_tr, X_val, y_cal_tr, y_val = tts(
    X_train, y_train,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_train,
)
print(f'Calibration train  : {len(X_cal_tr):,}')
print(f'Calibration holdout: {len(X_val):,}')

In [ ]:
# Re-fit CatBoost on the reduced calibration training set
print('Fitting CatBoost on calibration training split...')
cb_cal_base = catboost_classifier(
    iterations=500, learning_rate=0.05, depth=6, scale_pos_weight=spw, verbose=0
)
cb_cal_base.fit(
    X_cal_tr, y_cal_tr,
    cat_features=CATEGORICAL_FEATURES,
    eval_set=(X_val, y_val),
    verbose=False,
)

# Post-hoc calibration: Platt scaling and isotonic regression
cb_sigmoid  = calibrate(cb_cal_base, X_val, y_val, method='sigmoid')
cb_isotonic = calibrate(cb_cal_base, X_val, y_val, method='isotonic')
print('Calibrated models fitted.')

In [ ]:
# Calibration metrics on the test set
prob_uncal   = cb_cal_base.predict_proba(X_test)[:, 1]
prob_sigmoid = cb_sigmoid.predict_proba(X_test)[:, 1]
prob_isoton  = cb_isotonic.predict_proba(X_test)[:, 1]

print('=== Calibration Metrics (ECE / MCE) — Test Set ===')
for label, prob in [('Uncalibrated', prob_uncal), ('Sigmoid (Platt)', prob_sigmoid), ('Isotonic', prob_isoton)]:
    m = calibration_metrics(y_test, prob)
    print(f'  {label:20s}  ECE={m["ece"]:.4f}  MCE={m["mce"]:.4f}')

In [ ]:
# Reliability diagram comparing the three variants
models_for_diagram = {
    'Uncalibrated'   : (y_test, prob_uncal),
    'Sigmoid'        : (y_test, prob_sigmoid),
    'Isotonic'       : (y_test, prob_isoton),
}
fig = plot_reliability_diagram(models_for_diagram, n_bins=10, figsize=(13, 5))
plt.tight_layout()
plt.show()

## 8. Model Comparison Table

In [ ]:
# Collect calibration ECE for each model
def _ece(y, p):
    return calibration_metrics(y, p)['ece']

rows = []
model_registry = [
    ('Logistic Regression', lr_metrics, lr_prob),
    ('Random Forest',       rf_metrics, rf_prob),
    ('CatBoost',            cb_metrics, cb_prob),
    ('CatBoost + Sigmoid',  classification_report_dict(y_test, prob_sigmoid), prob_sigmoid),
    ('CatBoost + Isotonic', classification_report_dict(y_test, prob_isoton),  prob_isoton),
]

for name, metrics, prob in model_registry:
    rows.append({
        'Model'       : name,
        'ROC-AUC'     : round(metrics['roc_auc'], 4),
        'Brier Score' : round(metrics['brier_score'], 4),
        'Cohen Kappa' : round(metrics['cohen_kappa'], 4),
        'Gini'        : round(metrics['gini'], 4),
        'ECE'         : round(_ece(y_test, prob), 4),
    })

comparison_df = pd.DataFrame(rows).set_index('Model')
print('=== Model Comparison ===')
comparison_df

## 9. STP Routing

In [ ]:
# Apply STP routing using the best calibrated model (isotonic tends to best reduce ECE)
best_prob = prob_isoton  # swap to prob_sigmoid if isotonic over-fits

summary = stp_summary(best_prob, y_true=y_test)
print('=== STP Routing Summary (CatBoost + Isotonic) ===')
print(summary.to_string())

In [ ]:
# Visualise lane distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

lane_order = ['auto-settle', 'review', 'investigate']
colors_lane = ['#55A868', '#4C72B0', '#DD8452']

# Policy count by lane
axes[0].bar(
    lane_order,
    [summary.loc[l, 'n_policies'] for l in lane_order],
    color=colors_lane, edgecolor='white'
)
axes[0].set_title('Policy Count by STP Lane')
axes[0].set_ylabel('Number of Policies')
for i, l in enumerate(lane_order):
    n = summary.loc[l, 'n_policies']
    p = summary.loc[l, 'pct']
    axes[0].text(i, n + 100, f'{n:,}\n({p}%)', ha='center', va='bottom', fontsize=9)

# Claim rate by lane
if 'claim_rate' in summary.columns:
    axes[1].bar(
        lane_order,
        [summary.loc[l, 'claim_rate'] for l in lane_order],
        color=colors_lane, edgecolor='white'
    )
    axes[1].set_title('Observed Claim Rate by STP Lane')
    axes[1].set_ylabel('Claim Rate')
    for i, l in enumerate(lane_order):
        cr = summary.loc[l, 'claim_rate']
        axes[1].text(i, cr + 0.002, f'{cr:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 10. MLflow Logging

In [ ]:
# Log the best model (CatBoost + isotonic calibration) to MLflow
best_metrics = classification_report_dict(y_test, prob_isoton)
best_cal_metrics = calibration_metrics(y_test, prob_isoton)

mlflow.set_experiment('claims_classification')

with mlflow.start_run(run_name='catboost_calibrated'):
    # Log model parameters
    mlflow.log_params({
        'model'             : 'CatBoostClassifier',
        'calibration'       : 'isotonic',
        'iterations'        : 500,
        'learning_rate'     : 0.05,
        'depth'             : 6,
        'scale_pos_weight'  : round(spw, 4),
        'train_size'        : len(X_train),
        'test_size'         : len(X_test),
        'features'          : str(all_features),
        'cat_features'      : str(CATEGORICAL_FEATURES),
        'random_state'      : RANDOM_STATE,
    })

    # Log classification metrics
    mlflow.log_metrics({
        'roc_auc'           : round(best_metrics['roc_auc'], 4),
        'avg_precision'     : round(best_metrics['avg_precision'], 4),
        'brier_score'       : round(best_metrics['brier_score'], 4),
        'cohen_kappa'       : round(best_metrics['cohen_kappa'], 4),
        'gini'              : round(best_metrics['gini'], 4),
        'ece'               : round(best_cal_metrics['ece'], 4),
        'mce'               : round(best_cal_metrics['mce'], 4),
    })

    # Log STP routing stats as metrics
    for lane in ['auto-settle', 'review', 'investigate']:
        safe_lane = lane.replace('-', '_')
        mlflow.log_metric(f'stp_{safe_lane}_pct', float(summary.loc[lane, 'pct']))
        if 'claim_rate' in summary.columns:
            mlflow.log_metric(f'stp_{safe_lane}_claim_rate', float(summary.loc[lane, 'claim_rate']))

    run_id = mlflow.active_run().info.run_id

print(f'MLflow run logged. Run ID: {run_id}')
print(f'View at: mlflow ui  (default: http://localhost:5000)')